# Inspect MEI notation and facsimile links locally

**Workflow 1 — create and review MEI editions.** This notebook always renders the MEI notation. When facsimile records are available, it also links rendered measures to image zones. It is an inspection tool: it reads the MEI and optional image but does not modify either one.

Set `MEI_PATH` to any local MEI. Files without `<facsimile>` records open in score-only mode instead of failing. If `<facsimile>`, `<zone type="measure">`, and measure `@facs` links exist, the linked two-pane viewer is enabled automatically. Local image targets are embedded; HTTP(S) and data-URI targets are also supported.


In [ ]:
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Find this checkout whether Jupyter started in the root or notebooks/."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "camat").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from camat import launch_interactive_facsimile_viewer, read_facsimile_model

DEFAULT_CORPUS_MEI = (
    REPO_ROOT.parent
    / "camat_corpus"
    / "Demo"
    / "bsb00023199_00175_facs_zones.mei"
)

DEFAULT_SCORE_ONLY_MEI = REPO_ROOT / "camat" / "examples" / "duration_semantics.mei"
DEFAULT_MEI = DEFAULT_CORPUS_MEI if DEFAULT_CORPUS_MEI.is_file() else DEFAULT_SCORE_ONLY_MEI

# Override with any working file, for example:
# MEI_OVERRIDE = REPO_ROOT / "converted_mei" / "my_score.mei"
MEI_OVERRIDE = None
configured_mei = (
    MEI_OVERRIDE
    or os.environ.get("CAMAT_INSPECT_MEI")
    or os.environ.get("CAMAT_FACSIMILE_MEI")  # backwards-compatible name
    or DEFAULT_MEI
)
MEI_PATH = Path(configured_mei).expanduser().resolve()
if not MEI_PATH.is_file():
    raise FileNotFoundError(f"No MEI file at {MEI_PATH}")

MEI_PATH


## 1. Inspect available facsimile records

With `allow_missing_facsimile=True`, `read_facsimile_model` returns a score-only model when no usable facsimile records exist. When records are present, it validates the first surface, resolves its graphic, and checks every measure link. Unresolved `@facs` links still raise an error because they indicate inconsistent—not merely absent—editorial data.


In [ ]:
facsimile_model = read_facsimile_model(MEI_PATH, allow_missing_facsimile=True)
print(f"Viewer mode:   {facsimile_model['viewer_mode']}")
print(f"Measures:      {len(facsimile_model['measures'])}")
if facsimile_model['has_facsimile']:
    print(f"Graphic:       {facsimile_model['graphic_target']}")
    print(f"Image size:    {facsimile_model['image_width']} × {facsimile_model['image_height']}")
    print(f"Linked zones:  {len(facsimile_model['linked'])}")
    print(f"Missing @facs: {len(facsimile_model['missing_facs'])}")
else:
    print(f"Facsimile:     {facsimile_model['facsimile_status']}")
    print("The viewer will render the notation at full width.")


## 2. Configure rendering and reloading

`ALLOW_MISSING_FACSIMILE=True` enables the score-only fallback. In that mode, **Check facsimile** can detect records added later without rerendering unchanged notation. With a linked facsimile, **Reload zones** reuses cached score SVG when only coordinates or `@facs` links changed. **Reload score** also reruns Verovio after notation or layout changes.


In [ ]:
VEROVIO_INITIAL_PAGE = 1
VEROVIO_ORIENTATION = "portrait"  # "portrait" or "landscape"
VEROVIO_BREAKS = "encoded"        # "auto", "encoded", or "none"
VEROVIO_ADJUST_PAGE_HEIGHT = True

VIEWER_MAX_HEIGHT = 820
FACSIMILE_MAX_WIDTH = 600
ZONE_OPACITY = 0.18
SHOW_DIAGNOSTIC_TABLE = True
ALLOW_MISSING_FACSIMILE = True

AUTO_WATCH_MEI = False
AUTO_WATCH_MODE = "events"  # watchdog events; falls back to timed polling
WATCH_INTERVAL_SEC = 2.0
WATCH_SETTLE_SEC = 0.35
WATCH_DEBOUNCE_SEC = 0.4

VEROVIO_OPTIONS = {
    "footer": "none",
    "scale": 35,
    "svgViewBox": True,
}


## 3. Launch the adaptive viewer

Without facsimile records, the notation uses the full viewer width. With linked records, hover or click a measure in either pane to highlight its partner. Verovio score-page controls affect only the rendered notation; the facsimile remains the first surface declared in the MEI.


In [ ]:
viewer = launch_interactive_facsimile_viewer(
    MEI_PATH,
    repo_root=REPO_ROOT,
    viewer_id="camat-mei-facsimile-viewer",
    verovio_initial_page=VEROVIO_INITIAL_PAGE,
    verovio_orientation=VEROVIO_ORIENTATION,
    verovio_breaks=VEROVIO_BREAKS,
    verovio_adjust_page_height=VEROVIO_ADJUST_PAGE_HEIGHT,
    verovio_options=VEROVIO_OPTIONS,
    viewer_max_height=VIEWER_MAX_HEIGHT,
    facsimile_max_width=FACSIMILE_MAX_WIDTH,
    zone_opacity=ZONE_OPACITY,
    show_diagnostic_table=SHOW_DIAGNOSTIC_TABLE,
    allow_missing_facsimile=ALLOW_MISSING_FACSIMILE,
    auto_watch_mei=AUTO_WATCH_MEI,
    auto_watch_mode=AUTO_WATCH_MODE,
    watch_interval_sec=WATCH_INTERVAL_SEC,
    watch_settle_sec=WATCH_SETTLE_SEC,
    watch_debounce_sec=WATCH_DEBOUNCE_SEC,
)


## What was produced?

- an in-memory inspection model describing either score-only or linked-facsimile mode;
- one or more cached Verovio SVG pages;
- an interactive HTML score view, with a linked facsimile pane when records exist.

No edition data was written. An absent facsimile is allowed; inconsistent records such as unresolved `@facs` links remain visible errors to fix in the editorial workflow. Reload here after editing, then continue to CAMAT's parsing and representation workflow once the page is reviewed.
